In [ ]:
print("hi")

In [1]:
!pip3 install SQLAlchemy requests


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\parth-nic\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
# # post response python equivalent of this for parsing response json data to python dictionary
# import requests, json

# access_value = "1CF88F38959D49C11493AD6FAEF204CA"
# access_key = "80B7B95DC8CF961A1864328C59C7A33F"
# deptid = "HPD0009"
# mode = "O"

# url = f"https://genpmis.hp.nic.in/android/etransfer.aspx?access_value={access_value}&access_key={access_key}&deptid={deptid}&mode={mode}"
# response = requests.get(url)
# data = response.json()



# # in loop for each record in data['records'] create item obj 
#                     #     foreach (var record in records)
#                     # {
#                     #     var item = new TransfersDetails
#                     #     {
#                     #         DepartmentName = record["DepartmentName"]?.ToString(),
#                     #         orderName = HttpUtility.HtmlDecode(record["orderName"]?.ToString()),
#                     #         HyperLink = record["HyperLink"]?.ToString(),
#                     #         deptid = record["deptid"]?.ToString(),
#                     #         uploadDate = record["uploadDate"]?.ToString(),
#                     #         RefreshDate = DateTime.Now.ToString("dd-MM-yyyy HH:mm:ss")
#                     #     };
#                     #     transfersDatabase.AddRecords(item);
#                     # }

# from datetime import datetime
# def process_data(data):
#     # process the data as needed
    

#         for i in data.get('records', []):
#             item = {
#                 'DepartmentName': i['DepartmentName'],
#                 'orderName': i['orderName'],
#                 'HyperLink': i['HyperLink'],
#                 'deptid': i['deptid'],
#                 'uploadDate': i['uploadDate'],
#                 'RefreshDate': datetime.now().strftime("%d-%m-%Y %H:%M:%S")
#             }
#             # add item to database modal class in orm or list as needed

#         return item

# processed_data = process_data(data)
# processed_data



# # pip install SQLAlchemy

# # Orm block
# from sqlalchemy import create_engine, Column, Integer, String
# from sqlalchemy.ext.declarative import declarative_base
# from sqlalchemy.orm import sessionmaker
# Base = declarative_base()
# class TransfersDetails(Base):
#     __tablename__ = 'transfers_details'
#     id = Column(Integer, primary_key=True)
#     DepartmentName = Column(String)
#     orderName = Column(String)
#     HyperLink = Column(String)
#     deptid = Column(String)
#     uploadDate = Column(String)
#     RefreshDate = Column(String)
# # create database connection and session
# engine = create_engine
# ('sqlite:///transfers.db')
# Base.metadata.create_all(engine)
# Session = sessionmaker(bind=engine)
# session = Session()

# # process data and add to database
# def process_data(data):
#     for record in data.get('records', []):
#         item = TransfersDetails(
#             DepartmentName=record.get('DepartmentName'),
#             orderName=record.get('orderName'),
#             HyperLink=record.get('HyperLink'),
#             deptid=record.get('deptid'),
#             uploadDate=record.get('uploadDate'),
#             RefreshDate=datetime.now().strftime("%d-%m-%Y %H:%M:%S")
#         )
#         session.add(item)
#     session.commit()

# process_data(data)


import requests
import html
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

# 1. Database Setup
Base = declarative_base()

class TransfersDetails(Base):
    __tablename__ = 'transfers_details'
    id = Column(Integer, primary_key=True)
    DepartmentName = Column(String)
    orderName = Column(String)
    HyperLink = Column(String)
    deptid = Column(String)
    uploadDate = Column(String)
    RefreshDate = Column(String)

# Initialize Connection
engine = create_engine('sqlite:///transfers.db')
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# 2. Fetch Data
access_value = "1CF88F38959D49C11493AD6FAEF204CA"
access_key = "80B7B95DC8CF961A1864328C59C7A33F"
deptid = "HPD0009"
mode = "O"

url = f"https://genpmis.hp.nic.in/android/etransfer.aspx?access_value={access_value}&access_key={access_key}&deptid={deptid}&mode={mode}"

try:
    response = requests.get(url)
    response.raise_for_status() # Check for HTTP errors
    data = response.json()
except Exception as e:
    print(f"Error fetching data: {e}")
    data = {}

# 3. Process and Save
def process_data(json_data):
    # .get('records', []) handles missing keys safely
    records = json_data.get('records', [])
    
    for i in records:
        # html.unescape handles the C# HttpUtility.HtmlDecode equivalent
        item = TransfersDetails(
            DepartmentName = i['DepartmentName'],
            orderName      = html.unescape(i.get('orderName', '')),
            HyperLink      = i['HyperLink'],
            deptid         = i['deptid'],
            uploadDate     = i['uploadDate'],
            RefreshDate    = datetime.now().strftime("%d-%m-%Y %H:%M:%S")
        )
        session.add(item)
    
    session.commit()
    print(f"Successfully processed {len(records)} records.")

process_data(data)

C:\Users\parth-nic\AppData\Local\Temp\ipykernel_8876\1868164904.py:101: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


Successfully processed 1 records.


In [3]:
# Read all records from the database
# records = session.query(TransfersDetails).all()
records = session.query(TransfersDetails).filter_by(uploadDate="26/02/2015").all()

for r in records:
    print(f"ID: {r.id}")
    print(f"Department: {r.DepartmentName}")
    print(f"Order: {r.orderName}")
    print(f"Link: {r.HyperLink}")
    print(f"Dept ID: {r.deptid}")
    print(f"Upload Date: {r.uploadDate}")
    print(f"Refresh Date: {r.RefreshDate}")
    print("---")
    
# drop all table in  tranfers.db
Base.metadata.drop_all(engine)


ID: 1
Department: FISHERIES
Order: Transfer order of Smt. Sunita Devi                                                                                                                                                                                                                        
Link: https://genpmis.hp.nic.in/Aspx/Reports/RptNoticeBoardPDF.aspx?deptId=HPD0009&ListId=15415
Dept ID: HPD0009
Upload Date: 26/02/2015
Refresh Date: 09-03-2026 12:13:30
---


In [ ]:
import requests
from datetime import datetime
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import declarative_base, sessionmaker

# --- 1. Fetch data from API ---
access_value = "1CF88F38959D49C11493AD6FAEF204CA"
access_key = "80B7B95DC8CF961A1864328C59C7A33F"
deptid = "HPD0009"
mode = "O"

url = f"https://genpmis.hp.nic.in/android/etransfer.aspx?access_value={access_value}&access_key={access_key}&deptid={deptid}&mode={mode}"
response = requests.get(url)
data = response.json()

# --- 2. Define ORM model ---
Base = declarative_base()

class TransfersDetails(Base):
    __tablename__ = 'transfers_details'
    id = Column(Integer, primary_key=True)
    DepartmentName = Column(String)
    orderName = Column(String)
    HyperLink = Column(String)
    deptid = Column(String)
    uploadDate = Column(String)
    RefreshDate = Column(String)

# --- 3. Create DB engine & session (SAME LINE!) ---
engine = create_engine('sqlite:///transfers.db')
Base.metadata.create_all(engine)
Session = sessionmaker(bind=engine)
session = Session()

# --- 4. Process & insert records ---
def process_data(data):
    items = []
    # for record in data.get('records', []):  # data.get('records', []) to avoid KeyError if 'records' key is missing this is equivalent to data['records'] if 'records' in data else []
    for record in data["records"]:
        item = TransfersDetails(
            DepartmentName=record.get('DepartmentName'),
            orderName=record.get('orderName'),
            HyperLink=record.get('HyperLink'),
            deptid=record.get('deptid'),
            uploadDate=record.get('uploadDate'),
            RefreshDate=datetime.now().strftime("%d-%m-%Y %H:%M:%S")
        )
        session.add(item)
        items.append(item)
    session.commit()
    return items

# C#  equivalient of 4 process_Data

# public List<TransfersDetails> ProcessData(dynamic data)
# {
#     var items = new List<TransfersDetails>();
#     foreach (var record in data.records)
#     {
#         var item = new TransfersDetails
#         {
#             DepartmentName = record.DepartmentName,
#             orderName = HttpUtility.HtmlDecode(record.orderName),
#             HyperLink = record.HyperLink,
#             deptid = record.deptid,
#             uploadDate = record.uploadDate,
#             RefreshDate = DateTime.Now.ToString("dd-MM-yyyy HH:mm:ss")
#         };
#         session.Add(item);
#         items.Add(item);
#     }
#     session.Commit();
#     return items;
# }

processed_data = process_data(data)
print(f"Inserted {len(processed_data)} records")

In [ ]:
# Read all records from the database
# records = session.query(TransfersDetails).all()
records = session.query(TransfersDetails).filter_by(uploadDate="26/02/2015").all()

for r in records:
    print(f"ID: {r.id}")
    print(f"Department: {r.DepartmentName}")
    print(f"Order: {r.orderName}")
    print(f"Link: {r.HyperLink}")
    print(f"Dept ID: {r.deptid}")
    print(f"Upload Date: {r.uploadDate}")
    print(f"Refresh Date: {r.RefreshDate}")
    print("---")
    
# drop all table in  tranfers.db
Base.metadata.drop_all(engine)
